# 2. Spectroscopy

## 2.1 Pulsed Resonator Spectroscopy

In [ ]:
# frequency range of spectroscopy scan - around expected centre frequency as defined in qubit parameters
spec_range = 10e6
# how many frequency points to measure
spec_num = 51

# define sweep parameters for two qubits
freq_sweep_ST = LinearSweepParameter(
    uid="res_freq",
    start=qubit_parameters["ro_freq"] - spec_range / 2,
    stop=qubit_parameters["ro_freq"] + spec_range / 2,
    count=spec_num,
)

# take how many averages per point: 2^n_average
n_average = 13

# spectroscopy excitation pulse
readout_pulse_spec = pulse_library.const(
    length=qubit_parameters["ro_len_spec"], amplitude=qubit_parameters["ro_amp_spec"]
)

In [ ]:
# define the experiment with the frequency sweep relevant for qubit 0
exp_spec = res_spectroscopy(freq_sweep_ST, readout_pulse_spec, n_average)

# set signal calibration and signal map for experiment to qubit 0
exp_spec.set_calibration(res_spec_calib(freq_sweep_ST))
exp_spec.set_signal_map(MA_map)

In [ ]:
#compile experiment
exp_spec_comp = my_session.compile(exp_spec)

In [ ]:
# run the experiment on the open instrument session
ST_results = my_session.run(exp_spec_comp)

meas_ready()

In [ ]:
# get the measurement data returned by the instruments from the LabOne Q session
spec_res = ST_results.get_data("res_spec")

# define the frequency axis from the qubit parameters
spec_freq = lo_settings["ro_lo"] + ST_results.get_axis("res_spec")[0]

In [ ]:
# fit to asymmetric Lorentzian model 
full_result, quick_result, sweep_manager = fit_single_STS_wrapper(freqs=spec_freq, y_data=spec_res)
print(quick_result)

fig, ax = plot_single_STS_wrapper(sweep_man=sweep_manager)

#Change RO freq
qubit_parameters.update_parameter("ro_freq", quick_result['f0'] - lo_settings["ro_lo"])
print('RO Frequency:', qubit_parameters["ro_freq"]*1e-6, 'MHz')

In [ ]:
print('Readout frequency from fit: ', qubit_parameters["ro_freq"]*1e-6, 'MHz')

## 2.2 Pulsed Qubit Spectroscopy

### 2.2.1 Pulsed Measurements Presettings

In [ ]:
#USUALLY DON'T NEED TO BE USED!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#define shortcuts for lo settings

#lsg_q0 = my_setup.logical_signal_groups["q0"].logical_signals
#drive_Oscillator_q1 = lsg_q0["drive_line"].local_oscillator

In [ ]:
#manual settings of the readout and exitation pulse properties

#exitation:
qubit_parameters.update_parameter("qb_len_spec", 20e-6)
qubit_parameters.update_parameter("qb_amp_spec", 0.6)#<----------- Most important parameter for good Qubit Spectroscopy!!!

print('Qubit spectroscopy length:', qubit_parameters["qb_len_spec"]*1e6, 'mks')
print('Qubit spectroscopy amplitude:', qubit_parameters["qb_amp_spec"])

#readout
#qubit_parameters["ro_len"]=2e-6 #Not bigger that 2e-6!
#qubit_parameters["ro_amp"] = 0.2

print('Readout length:', qubit_parameters["ro_len"]*1e6, 'mks')
print('Readout amplitude:', qubit_parameters["ro_amp"])

#relaxation
# qubit_parameters.update_parameter('relax', 75e-6)
print('Relaxation time:', qubit_parameters['relax']*1e6, 'mks')

In [ ]:
#USUALLY DON'T NEED TO BE USED!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#manual preset of the qubit frequency
#qubit_parameters["qb_freq"] = -100e6

In [ ]:
#PULSE SETTINGS
# square pulse to excite the qubit
# square_pulse = pulse_library.const(
#     uid="const_iq",
#     length=qubit_parameters["qb_len_spec"],
#     amplitude=qubit_parameters["qb_amp_spec"],
# )

square_pulse = pulse_library.gaussian(
    uid="const_gauss",
    length=qubit_parameters["qb_len_spec"],
    amplitude=qubit_parameters["qb_amp_spec"],
)

# qubit readout pulse - here simple constant pulse
readout_pulse = pulse_library.const(
    uid="readout_pulse",
    length=qubit_parameters["ro_len"],
    amplitude=qubit_parameters["ro_amp"],
)
# integration weights for qubit measurement - here simple constant weights, i.e. all parts of the return signal are weighted equally
readout_weighting_function = pulse_library.const(
    uid="readout_weighting_function", length=qubit_parameters["ro_len"], amplitude=1.0
)

In [ ]:
readout_spec = {'readout_type': 'spectroscopy',
                'readout_pulse': readout_pulse,
                'readout_weighting_function': readout_weighting_function,
                'relax_time': qubit_parameters["relax"],
                'measure_freq': qubit_parameters["ro_freq"],
                'acquire_freq': qubit_parameters["ro_freq"],
                'readout_range': -25,
                'readout_delay': qubit_parameters['ro_int_delay']
               }

In [ ]:
print('Qubit frequency before spectroscopy: ', qubit_parameters["qb_freq"]*1e-6, 'MHz')

In [ ]:
# frequency range of spectroscopy scan - defined around expected qubit frequency as defined in qubit parameters
qspec_range = 10e6
# how many frequency points to measure
qspec_num = 101

#Set sweep with span around qubit frequency
freq_sweep_TT = LinearSweepParameter(
    uid="freq-qubit",
    start=qubit_parameters["qb_freq"] - qspec_range / 2,
    stop=qubit_parameters["qb_freq"] +  qspec_range / 2,
    count=qspec_num,
)

# #Set sweep in fixed range
# freq_sweep_TT = LinearSweepParameter(
#    uid="freq-qubit",
#    start=0e6,
#    stop=200e6,
#    count=qspec_num,
# )

# how many averages per point: 2^n_average
n_average = 12

In [ ]:
# qubit_parameters.update_parameter("qb_freq", 4.327e9 - lo_settings["qb_lo"]) 132.81175270395565
# qubit_parameters.update_parameter("qb_freq", 132.81175270395565e6)

In [ ]:
## configure oscillator settings for readout and acquisition - defined as calibration on device setup as persistent baseline settings

# readout pulse frequency resonant with the readout resonator
readout_Oscillator_q0.frequency = qubit_parameters["ro_freq"]
# change oscillator type to software to enable multiplexed qubit readout
readout_Oscillator_q0.modulation_type = ModulationType.SOFTWARE
# demodulation frequency same as readout pulse frequency
acquire_Oscillator_q0.frequency = qubit_parameters["ro_freq"]
acquire_Oscillator_q0.modulation_type = ModulationType.SOFTWARE

print('Readout frequency set to: ', qubit_parameters["ro_freq"]*1e-6, 'MHz')

In [ ]:
#Uncalibrated spectroscopy readout
exp_qspec = create_qubit_spec(freq_sweep_TT, square_pulse, readout_spec, n_average)

#Calibrated low-power readout
#exp_qspec = create_qubit_spec(freq_sweep_TT, square_pulse, readout_low, n_average)

In [ ]:
#compile experiment

exp_qspec_comp = my_session.compile(exp_qspec)
#show_pulse_sheet("Q spec", my_session.compiled_experiment)

In [ ]:
# run the experiment
qspec_results = my_session.run(exp_qspec_comp)

#meas_ready()

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("q0_spec")

# define a frequency axis from the parameters
qspec_freq = lo_settings["qb_lo"] + qspec_results.get_axis("q0_spec")[0]
#qspec_freq = drive_Oscillator_q1.frequency + my_results.get_axis("q0_spec")[0]

In [ ]:
# plot measurement data
fig, (ax1, ax2) = plt.subplots(2, 1, sharex = True, figsize=(12,8))
ax1.plot(qspec_freq / 1e9, abs(qspec_res), ".k", label = 'amp'),
ax2.plot(qspec_freq / 1e9, np.unwrap(np.angle(qspec_res)), ".k", label = 'pha')
ax1.set_ylabel("Amplitude, a.u.", fontsize = 14)
ax2.set_ylabel("Phase, rad", fontsize = 14)
ax2.set_xlabel("Frequency, GHz", fontsize = 14)
plt.suptitle('Pulse qubit spectroscopy', y = 0.92, fontsize = 14)

# increase number of plot points for smooth plotting of fit reults
freq_plot = np.linspace(qspec_freq[0], qspec_freq[-1], 5 * len(qspec_freq))

guess_freq = 3.8478e9

manual_freq = 3.8478e9
# ax1.vlines([manual_freq / 1e9], abs(qspec_res).min(), abs(qspec_res).max())
# ax2.vlines([manual_freq / 1e9], np.unwrap(np.angle(qspec_res)).min(), np.unwrap(np.angle(qspec_res)).max())

# fit measurement data - here assuming an inverted Lorentzian response
popt_pha, pcov_pha = fit_3DSpec(
   qspec_freq,
   #abs(qspec_res),
   np.unwrap(np.angle(qspec_res)),
   1e6,
   guess_freq,#qubit_parameters["qb0_freq"] + lo_settings["qb_lo"],
   10e3,
   off=2.0,
   plot=False,
   #bounds=[[5e5, 5.24e9, 0, 0.02], [3e6, 5.280e9, 3.0e5, 0.06]],
   #bounds=[[10e4, 5.210e9, 0, 0], [10e6, 5.240e9, 1e8, 2]],
)

# fit measurement data - here assuming an inverted Lorentzian response
popt_amp, pcov_amp = fit_3DSpec(
    qspec_freq,
    abs(qspec_res),
    #np.unwrap(np.angle(qspec_res)),
    5e5,
    guess_freq,#qubit_parameters["qb0_freq"] + lo_settings["qb_lo"],
    10.0e3,
    off=0.28,
    plot=False,
    #bounds=[[5e5, 5.24e9, 0, 0.02], [3e6, 5.280e9, 3.0e5, 0.06]],
    #bounds=[[10e4, 5.210e9, 0, 0], [10e6, 5.240e9, 1e8, 2]],
)

# plot fit results together with measurement data
ax1.plot(freq_plot / 1e9, func_lorentz(freq_plot, *popt_amp), "-r")
ax2.plot(freq_plot / 1e9, func_lorentz(freq_plot, *popt_pha), "-r")

plt.legend()

#save figure
# figname = 'Pulsed_TTS_2fig_'
# file_path = get_path_to_file(figname, '.png', sample_parameters)
# plt.savefig(file_path, dpi=600, format='png', metadata=None,
#         bbox_inches=None, pad_inches=0.1,
#         facecolor='auto', edgecolor='auto',
#         backend=None,
#        )

#print('Qubit frequency from fit:', round(popt[1]*1e-9, 3), 'GHz')

# update qubit parameters
qubit_parameters.update_parameter("qb_freq", popt_pha[1] - lo_settings["qb_lo"])
# qubit_parameters.update_parameter("qb_freq", popt_amp[1] - lo_settings["qb_lo"])
#qubit_parameters["qb1_freq"] = popt[1] - drive_Oscillator_q1.frequency
# qubit_parameters.update_parameter("qb_freq", manual_freq - lo_settings["qb_lo"])
print('Qubit parameter qb_freq:', qubit_parameters["qb_freq"]*1e-6, 'MHz')

In [ ]:
popt_pha[1]

In [ ]:
popt_amp[1]-4.2e9

In [ ]:
#qubit_parameters['qb_anharm'] = -(popt_amp[1] - lo_settings["qb_lo"]-qubit_parameters["qb_freq"])*2
#print('Qubit anharmonism: ', qubit_parameters['qb_anharm']*1e-6, 'MHz')

In [ ]:
Data_TT = {'qspec_freq': qspec_freq,
            'qspec_res': qspec_res
            }

Data_TT.update(qubit_parameters._params)
Data_TT.update(lo_settings._params)

file_name = 'TTS_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_TT)

In [ ]:
qubit_parameters

### 2.2.2 TTS with flux

In [ ]:
#Qubit spectroscopy
qspec_res_list = []
TTS_popt_list = []

In [ ]:
qubit_parameters['flux_lower_ss'] = -1.9e-05

In [ ]:
print(qubit_parameters['flux_lower_ss'])

In [ ]:
flux_sweep = np.linspace(-0.002, 0.002, 25)*1e-3
flux_points = qubit_parameters['flux_lower_ss']+flux_sweep
print(flux_points)

In [ ]:
for i in range(len(flux_sweep)):
    print('Measurement number:', i)
    volt_source.ramp(flux_points[i]*R, 10)
    print('Flux:', flux_points[i])
    
    # #Make the TTS
    my_results_TTS = my_session.run(exp_qspec_comp)
        
    qspec_res = my_results_TTS.get_data("q0_spec")
    qspec_freq = lo_settings["qb_lo"] + my_results_TTS.get_axis("q0_spec")[0]
        
    qspec_res_list.append(qspec_res)

    popt_pha, pcov_pha = fit_3DSpec(
           qspec_freq,
           #abs(qspec_res),
           np.unwrap(np.angle(qspec_res)),
           1e6,
           4.6e9,#qubit_parameters["qb0_freq"] + lo_settings["qb_lo"],
           10e3,
           off=1.84,
           plot=True,
           #bounds=[[5e5, 5.24e9, 0, 0.02], [3e6, 5.280e9, 3.0e5, 0.06]],
           #bounds=[[10e4, 5.210e9, 0, 0], [10e6, 5.240e9, 1e8, 2]],
    )

    TTS_popt_list.append(popt_pha)

In [ ]:
qspec_res_arr = np.array(qspec_res_list)
TTS_popt_arr = np.array(TTS_popt_list)

In [ ]:
TTS_popt_arr.shape

In [ ]:
plt.plot(flux_points, TTS_popt_arr[:,1])

In [ ]:
qubit_parameters['flux_lower_ss'] = -1.85e-05
volt_source.ramp(qubit_parameters['flux_lower_ss']*R, 10)

## 2.3 e-f Transition Spectroscopy

In [ ]:
# qubit_parameters.update_parameter('qb_anharm', 220e6)
#qubit_parameters["ro_len"]=2e-6
#qubit_parameters['qb_ef_amp_spec']=0.2
# qubit_parameters.update_parameter('qb_ef_amp_spec', 0.2)
#qubit_parameters['qb4_freq']=22e6

In [ ]:
#manual settings of the readout and exitation pulse properties

#exitation:
qubit_parameters.update_parameter("qb_len_spec", 20e-6)
qubit_parameters.update_parameter("qb_ef_amp_spec", 0.8)#<----------- Most important parameter for good Qubit Spectroscopy!!!

print('Qubit spectroscopy length:', qubit_parameters["qb_len_spec"]*1e6, 'mks')
print('Qubit spectroscopy ef amplitude:', qubit_parameters["qb_ef_amp_spec"])

In [ ]:
#qubit_parameters["qb_freq"]*1e-6

In [ ]:
qubit_parameters['qb_anharm']

In [1]:
#qubit_parameters["qb_freq"]+3.8e9-3.6344e9

In [ ]:
#qubit_parameters.update_parameter('qb_anharm', 213307499.87887526)

In [ ]:
# frequency range of spectroscopy scan - defined around expected qubit frequency as defined in qubit parameters
qspec_range = 20e6
# how many frequency points to measure
qspec_num = 501

# set up sweep parameters - qubit drive frequency
freq_sweep_ef = LinearSweepParameter(
    uid="freq-ef-qubit",
    start=qubit_parameters["qb_freq"] - qubit_parameters['qb_anharm'] - qspec_range / 2,
    stop=qubit_parameters["qb_freq"]  - qubit_parameters['qb_anharm'] +  qspec_range / 2,
    count=qspec_num,
)

# how many averages per point: 2^n_average
n_average = 14

# square pulse to excite the qubit
square_pulse = pulse_library.const(
    uid="const_iq",
    length=qubit_parameters["qb_len_spec"],
    amplitude=qubit_parameters["qb_ef_amp_spec"],
)

In [ ]:
# exp_ef_qspec = create_qubit_ef_spec(freq_sweep_ef, 
#                                     square_pulse, 
#                                     readout_spec, 
#                                     n_average)

exp_ef_qspec = create_qubit_ef_spec_prep(freq_sweep_ef, 
                                    x180,
                                    square_pulse, 
                                    readout_low, 
                                    n_average)

In [ ]:
#compile the experiment
exp_ef_qspec_comp = my_session.compile(exp_ef_qspec)
#show_pulse_sheet("Q spec", my_session.compiled_experiment)

In [ ]:
# run the experiment on qubit 0
ef_qspec_results = my_session.run(exp_ef_qspec_comp)

In [ ]:
# get measurement data returned by the instruments
ef_qspec_res = ef_qspec_results.get_data("q0_ef_spec")

# define a frequency axis from the parameters
#ef_qspec_freq = drive_Oscillator_q1.frequency + my_results.get_axis("q0_ef_spec")[0]
ef_qspec_freq = lo_settings["qb_lo"] + ef_qspec_results.get_axis("q0_ef_spec")[0]

In [ ]:
# plot measurement data
#%matplotlib inline
fig = plt.figure()
# plt.plot(ef_qspec_freq / 1e9, abs(ef_qspec_res), ".k")
plt.plot(ef_qspec_freq / 1e9, np.unwrap(np.angle(ef_qspec_res)), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")
plt.title('Pulse qubit ef-spectroscopy')

#xlim = [4.997, 5.003]
#plt.xlim(xlim)
#ylim = [0.032, 0.034]
#plt.ylim(ylim)

# increase number of plot points for smooth plotting of fit reults
freq_plot = np.linspace(ef_qspec_freq[0], ef_qspec_freq[-1], 5 * len(ef_qspec_freq))

# fit measurement data - here assuming an inverted Lorentzian response
popt, pcov = fit_3DSpec(
    ef_qspec_freq,
    #abs(ef_qspec_res),
    np.unwrap(np.angle(ef_qspec_res)),
    1e6,
    3.6345e9,#qubit_parameters["qb0_freq"] + lo_settings["qb_lo"],
    10,
    off=2.0,
    plot=False,
    #bounds=[[5e3, 6.2e9, 0, 0], [5e6, 6.3e9, 5.0e4, 0.06]],
    #bounds=[[10e4, 5.210e9, 0, 0], [10e6, 5.240e9, 1e8, 2]],
)
print(popt)
#%matplotlib QT
# plot fit results together with measurement data
plt.plot(freq_plot / 1e9, func_lorentz(freq_plot, *popt), "-r")
# update qubit parameters (uncomment only if fit converged!)
# qubit_parameters.update_parameter('qb_anharm', -(popt[1] - lo_settings["qb_lo"]-qubit_parameters["qb_freq"]))
#qubit_parameters['qb_anharm'] = -(popt[1] - lo_settings["qb_lo"]-qubit_parameters["qb_freq"])*2
# qubit_parameters.update_parameter('qb_anharm', -(3.635e9 - lo_settings["qb_lo"] - qubit_parameters["qb_freq"]))

#qubit_parameters['qb_anharm'] = popt[1] - qubit_parameters["qb1_freq"]
# qubit_parameters["qb4_freq"] = 3.636e9 - lo_settings["qb_lo"]
# print(qubit_parameters["qb4_freq"])
print('Qubit anharmonism: ', qubit_parameters['qb_anharm']*1e-6, 'MHz')

In [ ]:
#For manual setting of anharmonicity
#qubit_parameters['qb_anharm'] = 100e6

In [ ]:
#qubit_parameters["qb4_freq"] = popt[1] - lo_settings["qb_lo"]
qubit_parameters["qb4_freq"]-qubit_parameters["qb_freq"]

In [ ]:
qubit_parameters["qb4_freq"]

In [ ]:
3.8-3.63438

In [ ]:
Data_TT = {'qspec_freq': ef_qspec_freq,
            'qspec_res': ef_qspec_res
            }

Data_TT.update(qubit_parameters._params)
Data_TT.update(lo_settings._params)

file_name = 'ef_TTS_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_TT)